# Mushroom Dataset - Random Forest

## Contributors
- Liam: Random Forest training, tuning, validation, and metrics

This notebook trains and tunes a Random Forest independently from the baseline and XGBoost notebooks.

## Evaluation design

An 80/20 stratified split creates the training and holdout sets. Randomized tuning uses 5-fold cross-validation only on the training set; the holdout test set is used once for final validation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
df = pd.read_csv("mushroom_cleaned.csv")
target = "class"
X = df.drop(columns=target)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train shape: {X_train.shape}; test shape: {X_test.shape}")

In [ ]:
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, roc_auc_score
from pycaret.classification import add_metric, compare_models, finalize_model, predict_model, save_model, setup, tune_model

train_data = X_train.copy()
train_data[target] = y_train
setup(
    data=train_data,
    target=target,
    session_id=RANDOM_STATE,
    fold=5,
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    verbose=False,
    html=False,
)
add_metric(id="balanced_accuracy", name="Balanced Accuracy", score_func=balanced_accuracy_score, target="pred", greater_is_better=True)
rf_candidate = compare_models(include=["rf"], sort="Balanced Accuracy", n_select=1)
tuned_rf = tune_model(rf_candidate, optimize="Balanced Accuracy", fold=5, choose_better=True, verbose=False)
final_rf = finalize_model(tuned_rf)
rf_holdout = predict_model(final_rf, data=X_test.copy())
rf_labels = rf_holdout["prediction_label"]
rf_probabilities = final_rf.predict_proba(X_test)[:, 1]
rf_report = classification_report(y_test, rf_labels, target_names=["edible", "poisonous"], output_dict=True)
rf_metrics = {
    "model": "Tuned Random Forest",
    "accuracy": accuracy_score(y_test, rf_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, rf_labels),
    "poisonous_precision": precision_score(y_test, rf_labels, pos_label="p"),
    "poisonous_recall": recall_score(y_test, rf_labels, pos_label="p"),
    "poisonous_f1": f1_score(y_test, rf_labels, pos_label="p"),
    "roc_auc": roc_auc_score((y_test == "p").astype(int), rf_probabilities),
}
print(pd.Series(rf_metrics))
print("Confusion matrix [edible, poisonous]:\\n", confusion_matrix(y_test, rf_labels))
pd.DataFrame([rf_metrics]).to_csv("random_forest_metrics.csv", index=False)
Path("random_forest_metrics.json").write_text(pd.Series({**rf_metrics, "classification_report": rf_report}).to_json(indent=2), encoding="utf-8")
save_model(final_rf, "random_forest_final")
print("Saved Random Forest model and metrics")

## Conclusion

The Random Forest is a strong candidate when it improves balanced accuracy and poisonous recall over the baseline. Its saved metrics can be compared directly with XGBoost in the comparison notebook.